# Notebook 03 — Statistiques exploratoires et visualisations

## Objectif
Ce notebook produit les **statistiques descriptives** et les **graphiques** qui permettent d'avoir une vue d'ensemble des données :
- Nombre de prises de parole par parti et par bloc
- Évolution temporelle des débats sur les VSS
- Heatmap des mots-clés par parti
- Analyse du vocabulaire identitaire dans les discours VSS
- Synthèses LLM par bloc (via le serveur Ollama de l'ENSAE)

## Entrées
- `df_vss_propre.pkl` (notebook 02)
- `df_global.pkl` (notebook 02) — optionnel, pour la normalisation

## Sorties
- Graphiques affichés dans le notebook
- Fichiers texte par parti (dossier `analyses_par_partis/`)
- Synthèses par bloc (fichier `syntheses_blocs_VSS.txt`)

## 1. Imports et chargement des données

In [ ]:
from config import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import os

# Chargement des données
df_vss = pd.read_pickle(CHEMIN_DF_VSS_PROPRE)
print(f"✅ {len(df_vss)} prises de parole VSS chargées.")
print(f"   Blocs : {df_vss['bloc'].value_counts().to_dict()}")

# Chargement optionnel de df_global pour la normalisation
if os.path.exists(CHEMIN_DF_GLOBAL):
    df_global = pd.read_pickle(CHEMIN_DF_GLOBAL)
    print(f"   {len(df_global)} prises de parole globales chargées (pour normalisation).")
    HAS_GLOBAL = True
else:
    print("   ⚠️ df_global.pkl introuvable — les statistiques normalisées ne seront pas disponibles.")
    HAS_GLOBAL = False

## 2. Top des partis par volume de prises de parole VSS

Ce graphique montre quels partis interviennent le plus souvent sur les sujets VSS. Attention : un parti qui parle beaucoup en général parlera aussi plus souvent des VSS, d'où l'intérêt de la normalisation (section suivante).

In [ ]:
vss_counts = df_vss['nom_parti'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(
    x=vss_counts.values, y=vss_counts.index,
    hue=vss_counts.index, palette="viridis", legend=False
)
plt.title("Top 10 des partis par nombre de prises de parole VSS")
plt.xlabel("Nombre de paragraphes contenant un mot-clé VSS")
plt.ylabel("Parti politique")
plt.tight_layout()
plt.show()

## 3. Statistiques normalisées (si df_global disponible)

Pour comparer équitablement les partis, on calcule la **proportion** de prises de parole consacrées aux VSS (nombre de PdP VSS / nombre total de PdP). Cela neutralise l'effet de taille des groupes parlementaires.

In [ ]:
if HAS_GLOBAL:
    # Normalisation par parti
    total_par_parti = df_global.groupby('nom_parti').size().reset_index(name='total')
    vss_par_parti = df_vss.groupby('nom_parti').size().reset_index(name='vss')
    stats = total_par_parti.merge(vss_par_parti, on='nom_parti', how='left')
    stats['vss'] = stats['vss'].fillna(0)
    stats['proportion_vss'] = stats['vss'] / stats['total'] * 100
    stats = stats.sort_values('proportion_vss', ascending=False).head(15)

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=stats, x='proportion_vss', y='nom_parti',
        hue='nom_parti', palette="viridis", legend=False
    )
    for i, row in enumerate(stats.itertuples()):
        plt.text(row.proportion_vss + 0.05, i,
                 f"{row.proportion_vss:.1f}% ({int(row.vss)}/{int(row.total)})",
                 va='center', fontsize=9)
    plt.title("Proportion de prises de parole consacrées aux VSS par parti")
    plt.xlabel("% des prises de parole sur les VSS")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("⏩ Statistiques normalisées non disponibles (df_global manquant).")

## 4. Évolution temporelle des débats VSS

In [ ]:
df_vss['date'] = pd.to_datetime(df_vss['date'])
evolution = df_vss.resample('ME', on='date').size()

plt.figure(figsize=(14, 6))
evolution.plot(kind='line', marker='o', color='teal', markersize=4)
plt.title("Évolution mensuelle des prises de parole sur les VSS")
plt.xlabel("Temps")
plt.ylabel("Nombre de paragraphes")
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 5. Heatmap des mots-clés par parti

Cette heatmap montre quels **types de mots-clés** chaque parti utilise le plus. Par exemple, un parti qui parle beaucoup d'"avortement" mais peu de "harcèlement" a un profil thématique différent d'un parti qui fait l'inverse.

In [ ]:
# Liste de mots-clés pour la heatmap (version élargie)
mots_heatmap = [
    "viol", "sexis", "féminicide", "harcèl", "misogyn",
    "sexuel", "machis", "égalité", "discrimin", "agress", "consent",
    "stéréotyp", "inégal", "domination", "patriarc", "genre", "quota",
    "lgbt", "transphobi", "homophobi", "non-binair", "mixité",
    "avortement", "ivg", "contraception",
]

for mot in mots_heatmap:
    df_vss[mot] = df_vss['texte'].str.contains(mot, case=False, na=False)

heatmap_data = df_vss.groupby('nom_parti')[mots_heatmap].sum()
heatmap_data = heatmap_data[heatmap_data.sum(axis=1) > 10]

plt.figure(figsize=(16, 10))
sns.heatmap(heatmap_data, annot=True, cmap="YlGnBu", fmt='g',
            cbar_kws={'label': 'Nombre de mentions'})
plt.title("Répartition des mots-clés VSS par parti politique")
plt.tight_layout()
plt.show()

## 6. Vocabulaire identitaire dans les débats VSS

On mesure ici la **proportion de phrases** contenant du vocabulaire identitaire/migratoire (immigration, frontière, clandestin, islam...) dans les discours sur les VSS. C'est un premier indicateur de la présence d'un "cadrage fémonationaliste".

In [ ]:
mots_id_graph = [
    "immigration", "frontière", "clandestin", "identité",
    "étranger", "expulsion", "oqtf", "territoire",
    "illégal", "illégaux", "confession", "islam"
]

pattern_id = re.compile(r'(?i)\b(' + '|'.join(mots_id_graph) + r')\w*\b')

df_tmp = df_vss.copy()
df_tmp['texte'] = df_tmp['texte'].fillna('').astype(str)
df_tmp['phrases'] = df_tmp['texte'].str.split(r'[.!?]+')
df_tmp['phrases'] = df_tmp['phrases'].apply(lambda l: [p.strip() for p in l if p.strip()])
df_tmp['total_phrases'] = df_tmp['phrases'].str.len()
df_tmp['phrases_id'] = df_tmp['phrases'].apply(
    lambda phrases: sum(1 for p in phrases if pattern_id.search(p))
)

# Par bloc
agg_bloc = df_tmp.groupby('bloc')[['phrases_id', 'total_phrases']].sum()
agg_bloc['pourcentage'] = (agg_bloc['phrases_id'] / agg_bloc['total_phrases'] * 100).fillna(0)
agg_bloc = agg_bloc.sort_values('pourcentage', ascending=False)
agg_bloc = agg_bloc[agg_bloc['pourcentage'] > 0]

plt.figure(figsize=(10, 6))
couleurs = [COULEURS_BLOCS.get(b, 'gray') for b in agg_bloc.index]
sns.barplot(x=agg_bloc['pourcentage'].values, y=agg_bloc.index,
            hue=agg_bloc.index, palette=couleurs, legend=False)
plt.title("Proportion de phrases contenant du vocabulaire identitaire\ndans les discours sur les VSS, par bloc")
plt.xlabel("% de phrases concernées")
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 7. Export des textes par parti (pour analyse LLM)

On sauvegarde les textes de chaque parti dans des fichiers `.txt` séparés, utilisables comme entrée pour les synthèses LLM.

In [ ]:
output_dir = os.path.join(DOSSIER_ANALYSES, "textes_par_parti")
os.makedirs(output_dir, exist_ok=True)

for parti, group in df_vss.groupby("nom_parti"):
    nom_propre = "".join(x for x in str(parti) if x.isalnum() or x in "._- ").strip()
    with open(os.path.join(output_dir, f"{nom_propre}.txt"), "w", encoding="utf-8") as f:
        for discours in group["texte"]:
            f.write(str(discours) + "\n\n---\n\n")

print(f"✅ Fichiers texte créés pour {len(df_vss['nom_parti'].unique())} partis dans {output_dir}")

## 8. Synthèses LLM par bloc (Mistral via Ollama ENSAE)

On utilise une approche **Map-Reduce** :
1. **Map** : chaque bloc de textes est découpé en chunks de ~1200 mots, chaque chunk est résumé par Mistral
2. **Reduce** : les résumés intermédiaires sont fusionnés en une synthèse finale

> **⏱ Temps estimé** : ~30-60 min (dépend du serveur ENSAE)
> **⚠️ Nécessite** le serveur Ollama de l'ENSAE — ne fonctionnera pas hors du réseau ENSAE.

In [ ]:
import time, requests, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

fichier_syntheses = os.path.join(DOSSIER_ANALYSES, "syntheses_blocs_VSS.txt")

# --- CACHE : Si le fichier existe et contient tous les blocs, on saute ---
blocs_existants = set()
if os.path.exists(fichier_syntheses):
    with open(fichier_syntheses, "r", encoding="utf-8") as f:
        contenu = f.read()
    for bloc in ORDRE_BLOCS:
        if f"=== {bloc.upper()} ===" in contenu:
            blocs_existants.add(bloc)

blocs_a_faire = [b for b in ORDRE_BLOCS if b not in blocs_existants]

if not blocs_a_faire:
    print(f"⏩ Toutes les synthèses existent déjà dans {fichier_syntheses}")
else:
    print(f"🔄 Synthèses à générer : {blocs_a_faire}")

    def appeler_llm(prompt, role_systeme, max_retries=5):
        payload = {
            "model": "mistral",
            "messages": [
                {"role": "system", "content": role_systeme},
                {"role": "user", "content": prompt}
            ],
            "stream": False,
            "options": {"temperature": 0.0}
        }
        for attempt in range(max_retries):
            try:
                r = requests.post(URL_OLLAMA, json=payload, timeout=180, verify=False)
                r.raise_for_status()
                return r.json()["message"]["content"]
            except Exception as e:
                print(f"   Tentative {attempt+1}/{max_retries} échouée : {e}")
                time.sleep(30)
        return ""

    def decouper_textes(textes, max_mots=1200):
        mots = " ".join(textes).split()
        return [" ".join(mots[i:i+max_mots]) for i in range(0, len(mots), max_mots)]

    prompt_map = (
        "Tu es un chercheur en sociologie politique neutre et rigoureux. "
        "Synthétise des discours de l'Assemblée Nationale. "
        "N'utilise QUE les informations présentes dans le texte. "
        "Conserve le ton politique des orateurs."
    )
    prompt_reduce = (
        "Tu es un politologue expert. À partir des notes de synthèse, "
        "rédige un résumé cohérent décrivant la vision de ce bloc sur les VSS. "
        "Conserve le vocabulaire spécifique utilisé."
    )

    for bloc in blocs_a_faire:
        print(f"\n⚙️ Bloc : {bloc}...")
        discours = df_vss[df_vss['bloc'] == bloc]['texte'].tolist()
        if not discours:
            continue

        chunks = decouper_textes(discours)
        print(f"   {len(chunks)} chunks à analyser")

        resumes = []
        for chunk in tqdm(chunks, desc=f"Map {bloc}"):
            r = appeler_llm(
                f"Extrait du bloc '{bloc}' sur les VSS :\n\n{chunk}\n\nRésume en points clés.",
                prompt_map
            )
            if r:
                resumes.append(r)
            time.sleep(5)

        # Reduce itératif
        while len(resumes) > 8:
            nouveaux = []
            for i in range(0, len(resumes), 8):
                lot = "\n\n---\n\n".join(resumes[i:i+8])
                r = appeler_llm(f"Synthèse groupée pour '{bloc}':\n\n{lot}", prompt_reduce)
                if r:
                    nouveaux.append(r)
                time.sleep(5)
            resumes = nouveaux

        toutes_notes = "\n\n---\n\n".join(resumes)
        synthese = appeler_llm(
            f"Notes finales pour '{bloc}':\n\n{toutes_notes}\n\nRédige la synthèse (3 paragraphes).",
            prompt_reduce
        )

        with open(fichier_syntheses, "a", encoding="utf-8") as f:
            f.write(f"=== {bloc.upper()} ===\n{synthese}\n\n")
        print(f"   💾 Synthèse sauvegardée.")

    print(f"\n✅ Toutes les synthèses sont dans {fichier_syntheses}")